# Entrenament model reconeixement de cotxes
**Dataset:** Stanford Cars (16.185 imatges, 196 classes marca+model+any)
**Model:** EfficientNet-B3
**GPU:** T4 (Colab gratuït)
**Temps estimat:** 1-2 hores

### Com usar:
1. `Runtime → Change runtime type → T4 GPU`
2. Executa les cel·les en ordre
3. Al final descarrega `model_cotxes.pth`

In [ ]:
!pip install -q datasets timm torchvision Pillow tqdm

In [ ]:
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Cap GPU - ves a Runtime > Change runtime type > T4"}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from datasets import load_dataset
print('Descarregant Stanford Cars Dataset (~1.8GB)...')
dataset = load_dataset('tanganke/stanford_cars')
class_names = dataset['train'].features['label'].names
num_classes = len(class_names)
print(f'Train: {len(dataset["train"])} | Test: {len(dataset["test"])} | Classes: {num_classes}')
print('Exemples:', class_names[:5])

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

IMG_SIZE = 300
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CarsDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = hf_dataset
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return self.transform(item['image'].convert('RGB')), item['label']

train_loader = DataLoader(CarsDataset(dataset['train'], train_transforms), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(CarsDataset(dataset['test'],  val_transforms),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Batches entrenament: {len(train_loader)} | validacio: {len(val_loader)}')

In [ ]:
import timm, torch.nn as nn
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
model = model.to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametres: {params:,} | Device: {device}')

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

EPOCHS = 20
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
history = {'loss': [], 'acc': []}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = model(images).max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_acc = 100. * correct / total
    avg_loss = running_loss / len(train_loader)
    scheduler.step()
    history['loss'].append(avg_loss)
    history['acc'].append(val_acc)
    print(f'Epoch {epoch+1:02}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc: {val_acc:.2f}%')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'class_names': class_names, 'num_classes': num_classes, 'val_acc': val_acc}, 'model_cotxes.pth')
        print(f'  Nou millor model guardat! Acc: {val_acc:.2f}%')

print(f'Millor precisio final: {best_acc:.2f}%')

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['loss']); ax1.set_title('Perdua entrenament'); ax1.set_xlabel('Epoch')
ax2.plot(history['acc'], color='green'); ax2.set_title('Precisio validacio (%)'); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig('resultats.png'); plt.show()
print(f'Millor precisio: {best_acc:.2f}%')

In [ ]:
# Prova amb una foto de buscocotxe.ad
import requests
from PIL import Image
from io import BytesIO

def identificar(url_o_ruta, top_k=3):
    if url_o_ruta.startswith('http'):
        img = Image.open(BytesIO(requests.get(url_o_ruta).content)).convert('RGB')
    else:
        img = Image.open(url_o_ruta).convert('RGB')
    tensor = val_transforms(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)
        top_probs, top_idxs = probs.topk(top_k)
    print('Top prediccions:')
    for prob, idx in zip(top_probs[0], top_idxs[0]):
        print(f'  {class_names[idx]:55s} {prob.item()*100:.1f}%')

identificar('https://www.buscocotxe.ad/uploads/fotos_items/589619/1775942632_img_7378.jpeg')

In [ ]:
from google.colab import files
files.download('model_cotxes.pth')
print('Model descarregat! Guarda-lo a /Users/nur/GitHub/')